# Options Catalog Sweep

This notebook validates representative public options across analytical and file-to-file conversions.

In [ ]:
from pathlib import Path

import schema_sanitizer as ss

out_dir = Path("files/05_full_options_catalog_sweep")
out_dir.mkdir(parents=True, exist_ok=True)
input_csv = out_dir / "input_events.csv"
input_jsonl = out_dir / "input_events.jsonl"
input_csv.write_text(
    "id,flag,ts\n1,yes,2024/01/02 03:04:05\n2,no,2024/01/03 04:05:06\n", encoding="utf-8"
)
input_jsonl.write_text(
    '{"id": 1, "payload": {"x": 1}}\n{"id": 2, "payload": {"x": 2}}\n', encoding="utf-8"
)

In [ ]:
OPTION_SETS = [
    {},
    {"true_tokens": ("yes",), "false_tokens": ("no",)},
    {"custom_timestamp_patterns": (r"(\d{4})/(\d{2})/(\d{2}) (\d{2}):(\d{2}):(\d{2})",)},
    {"on_error": "emit_null_row"},
    {"arrow_max_depth": 1},
    {"parquet_max_depth": 1},
]
for options in OPTION_SETS:
    ss.to_pyarrow(input_csv, input_format="csv", **options)
print("validated option sets:", len(OPTION_SETS))

In [ ]:
options = OPTION_SETS[1]
result = ss.to_pyarrow(input_csv, input_format="csv", **options)
print(result.clean_data.schema)
print(result.stats)

In [ ]:
outputs = {
    "csv": ss.to_csv,
    "jsonl": ss.to_jsonl,
    "parquet": ss.to_parquet,
}
for suffix, convert in outputs.items():
    out = out_dir / f"output_events.{suffix}"
    convert(input_jsonl, out, input_format="jsonl")
    print(out, out.exists())